# SFIL Text2SQL SFT Pipeline (Adapted)

"
"This notebook is adapted to the project pipeline in this repository:
"
"- generation (`script 03`)
"
"- deterministic validation (`script 04`)
"
"- LLM judge (`script 05/06`)

"
"It builds a fine-tuning dataset from per-model judge outputs (`data/intermediate/by_model/*`) and provides an optional Unsloth + LoRA training flow for Qwen-style models.


In [ ]:
# Imports and root resolution
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict


def resolve_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for _ in range(8):
        if (cur / "assets").exists() and (cur / "data").exists():
            return cur
        if cur.parent == cur:
            break
        cur = cur.parent
    raise FileNotFoundError("Could not resolve repository root from current directory.")


ROOT = resolve_repo_root(Path.cwd())
print("ROOT =", ROOT)


In [ ]:
# Configuration
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BY_MODEL_DIR = ROOT / "data" / "intermediate" / "by_model"
ASSETS_DIR = ROOT / "assets"
OUTPUT_DIR = ROOT / "data" / "finetune"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ALIASES = [
    "deepseek_r1",
    "gpt_oss_120b",
    "mistral_small_24b",
    "qwen_7b",
]

SOURCE_KIND = "pass"  # "pass" (recommended) or "judged"
PASS_SCORE_THRESHOLD = 4.0

SCHEMA_SQL_PATH = ASSETS_DIR / "schema_sqlite.sql"
SCHEMA_GUIDE_PATH = ASSETS_DIR / "schema_guide.yaml"
BUSINESS_NARRATIVE_PATH = ASSETS_DIR / "business_context_sfil.md"

print("Using aliases:", MODEL_ALIASES)
print("Source kind:", SOURCE_KIND)


In [ ]:
# Load prompt context from assets
def read_text(path: Path, default: str = "") -> str:
    if not path.exists():
        return default
    return path.read_text(encoding="utf-8").strip()

SCHEMA_SQL_TEXT = read_text(SCHEMA_SQL_PATH)
SCHEMA_GUIDE_TEXT = read_text(SCHEMA_GUIDE_PATH)
BUSINESS_NARRATIVE_TEXT = read_text(BUSINESS_NARRATIVE_PATH)

print("schema_sql chars:", len(SCHEMA_SQL_TEXT))
print("schema_guide chars:", len(SCHEMA_GUIDE_TEXT))
print("business_narrative chars:", len(BUSINESS_NARRATIVE_TEXT))


In [ ]:
# Load judge outputs from by_model
def load_jsonl(path: Path) -> list[dict]:
    rows = []
    if not path.exists():
        return rows
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_model_rows(alias: str, source_kind: str = "pass") -> pd.DataFrame:
    model_dir = BY_MODEL_DIR / alias
    pass_path = model_dir / "ttsql_candidates_judge_pass.jsonl"
    judged_path = model_dir / "ttsql_candidates_judged.jsonl"

    if source_kind == "pass":
        rows = load_jsonl(pass_path)
    elif source_kind == "judged":
        rows = load_jsonl(judged_path)
        rows = [r for r in rows if bool(r.get("judge_pass", False))]
    else:
        raise ValueError(f"Unknown source_kind={source_kind}")

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    # Safety filter if judged file is loaded directly.
    if "judge_score" in df.columns:
        df = df[(df["judge_score"].fillna(0) >= PASS_SCORE_THRESHOLD)]
    if "judge_verdict" in df.columns:
        df = df[df["judge_verdict"].fillna("") == "pass"]

    df["source_model_alias"] = alias
    return df


frames = []
for alias in MODEL_ALIASES:
    df_alias = load_model_rows(alias, source_kind=SOURCE_KIND)
    print(f"{alias}: {len(df_alias)} rows")
    if not df_alias.empty:
        frames.append(df_alias)

if not frames:
    raise RuntimeError("No rows loaded from by_model outputs.")

raw_df = pd.concat(frames, ignore_index=True)
print("Total loaded rows:", len(raw_df))
raw_df.head(3)


In [ ]:
# Clean + deduplicate
df = raw_df.copy()

required_cols = ["question_fr", "sql"]
for col in required_cols:
    if col not in df.columns:
        raise KeyError(f"Missing required column: {col}")

df = df.dropna(subset=required_cols).copy()
df["question_fr"] = df["question_fr"].astype(str).str.strip()
df["sql"] = df["sql"].astype(str).str.strip()
df = df[(df["question_fr"] != "") & (df["sql"] != "")]

# Keep one sample per (question, sql) pair.
df = df.drop_duplicates(subset=["question_fr", "sql"]).reset_index(drop=True)

if "difficulty" not in df.columns:
    df["difficulty"] = "unknown"

print("Rows after cleaning/dedup:", len(df))
print("Rows by source model:")
print(df.groupby("source_model_alias").size().sort_values(ascending=False))
print("\nRows by difficulty:")
print(df.groupby("difficulty").size().sort_values(ascending=False))


In [ ]:
# Build training prompts and HF splits
PROMPT_CONTEXT = (
    "### SFIL Business Context\n"
    + BUSINESS_NARRATIVE_TEXT[:12000]
    + "\n\n### SQLite Schema (DDL)\n"
    + SCHEMA_SQL_TEXT[:12000]
    + "\n\n### Schema Guide Excerpt\n"
    + SCHEMA_GUIDE_TEXT[:12000]
)


def build_prompt(question: str) -> str:
    system_message = (
        "You are an expert Text-to-SQL assistant for SFIL municipal finance analysis.\n"
        "Return ONLY one SQLite SELECT query, with no explanation and no markdown.\n"
        "Rules:\n"
        "- Use only schema elements from the provided context.\n"
        "- Read-only SQL only (SELECT / WITH ... SELECT).\n"
        "- No INSERT/UPDATE/DELETE/DDL/PRAGMA.\n"
        + PROMPT_CONTEXT
    )
    return (
        f"<|im_start|>system\n{system_message}<|im_end|>\n"
        f"<|im_start|>user\n{question}<|im_end|>\n"
        "<|im_start|>assistant\n"
    )


df_ft = df.copy()
df_ft["prompt"] = df_ft["question_fr"].apply(build_prompt)
df_ft["answer"] = df_ft["sql"]

full_dataset = Dataset.from_pandas(
    df_ft[["question_fr", "answer", "prompt", "source_model_alias", "difficulty"]],
    preserve_index=False,
)

train_split = full_dataset.train_test_split(test_size=0.2, seed=SEED)
train_dataset = train_split["train"]
rest_split = train_split["test"].train_test_split(test_size=0.5, seed=SEED)
val_dataset = rest_split["train"]
test_dataset = rest_split["test"]

dataset = DatasetDict(train=train_dataset, validation=val_dataset, test=test_dataset)
print({k: len(v) for k, v in dataset.items()})


def format_prompt_train(examples):
    return [p + a.strip() + "\n<|im_end|>" for p, a in zip(examples["prompt"], examples["answer"])]


def format_prompt_eval(example):
    return example["prompt"]


## Optional SFT Training (Unsloth + LoRA)

This section is intentionally disabled by default (`ENABLE_TRAINING = False`).
Enable it only when your GPU environment and dependencies are ready.


In [ ]:
# Optional training configuration
ENABLE_TRAINING = True

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"
MAX_SEQ_LENGTH = 4096
LOAD_IN_4BIT = True

PER_DEVICE_BATCH_SIZE = 2
GRAD_ACC_STEPS = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS = 2

ADAPTER_OUTPUT_DIR = OUTPUT_DIR / "qwen_sft_adapter"
RUN_OUTPUT_DIR = OUTPUT_DIR / "runs" / "qwen_sft"

print("Training enabled:", ENABLE_TRAINING)
print("Model:", MODEL_ID)
print("Adapter output:", ADAPTER_OUTPUT_DIR)


In [ ]:
# Optional training execution
if ENABLE_TRAINING:
    import gc
    import inspect
    import torch
    from transformers import TrainingArguments
    from trl import SFTTrainer
    from unsloth import FastLanguageModel

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=torch.bfloat16,
        load_in_4bit=LOAD_IN_4BIT,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = FastLanguageModel.get_peft_model(
        model,
        r=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=64,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
    )

    training_args = TrainingArguments(
        output_dir=str(RUN_OUTPUT_DIR),
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACC_STEPS,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_EPOCHS,
        logging_steps=20,
        save_strategy="epoch",
        bf16=True,
        fp16=False,
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        report_to="none",
    )

    sig = inspect.signature(SFTTrainer.__init__)
    trainer_kwargs = dict(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        formatting_func=format_prompt_train,
        packing=True,
    )

    if "tokenizer" in sig.parameters:
        trainer_kwargs["tokenizer"] = tokenizer
    elif "processing_class" in sig.parameters:
        trainer_kwargs["processing_class"] = tokenizer

    if "max_seq_length" in sig.parameters:
        trainer_kwargs["max_seq_length"] = MAX_SEQ_LENGTH

    trainer = SFTTrainer(**trainer_kwargs)

    gc.collect()
    torch.cuda.empty_cache()

    print("Starting SFT training...")
    trainer.train()
    trainer.save_model(str(ADAPTER_OUTPUT_DIR))
    print("Adapter saved to:", ADAPTER_OUTPUT_DIR)
else:
    print("Training skipped. Set ENABLE_TRAINING = True to run SFT.")


In [ ]:
# Export prepared splits for reproducibility / handoff
EXPORT_DIR = OUTPUT_DIR / "prepared"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

for split_name, ds in dataset.items():
    split_df = ds.to_pandas()

    jsonl_path = EXPORT_DIR / f"sft_{split_name}.jsonl"
    csv_path = EXPORT_DIR / f"sft_{split_name}.csv"

    split_df.to_json(jsonl_path, orient="records", lines=True, force_ascii=False)
    split_df.to_csv(csv_path, index=False)

    print(f"[{split_name}] rows={len(split_df)} -> {jsonl_path.name}, {csv_path.name}")


## Inference + Judge + Figures (Base vs Fine-Tuned)

This section runs the full comparison pipeline for 5 training sources:
- `all_models`
- `deepseek_r1`
- `gpt_oss_120b`
- `mistral_small_24b`
- `qwen_7b`

For each source:
1. Build splits
2. Generate SQL with base model
3. Fine-tune on source train split and generate SQL
4. Judge execution/logic/fit with DeepSeek
5. Save results and plot comparison figures


In [ ]:
# Install extras if needed (Colab)
# !pip install -q openai matplotlib tqdm

import os
import gc
import re
import json
import inspect
import random
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from openai import OpenAI

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

assert os.getenv("DEEPSEEK_API_KEY"), "Missing DEEPSEEK_API_KEY in environment"

judge_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1",
)

JUDGE_MODEL_ID = "deepseek-chat"
DB_SQLITE_PATH = ROOT / "data" / "raw" / "db.sqlite"

FT_COMPARE_DIR = OUTPUT_DIR / "ft_compare"
ADAPTERS_DIR = OUTPUT_DIR / "adapters_by_source"
FT_COMPARE_DIR.mkdir(parents=True, exist_ok=True)
ADAPTERS_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_ORDER = ["all_models", "deepseek_r1", "gpt_oss_120b", "mistral_small_24b", "qwen_7b"]

MAX_EVAL_SAMPLES = 80
INFER_BATCH_SIZE = 4

print("Judge model:", JUDGE_MODEL_ID)
print("DB:", DB_SQLITE_PATH)
print("Sources:", SOURCE_ORDER)


In [ ]:
# Build 5 source datasets from df_ft
source_dfs = {"all_models": df_ft.copy()}
for alias in sorted(df_ft["source_model_alias"].dropna().unique()):
    source_dfs[alias] = df_ft[df_ft["source_model_alias"] == alias].copy()

for alias in SOURCE_ORDER:
    if alias not in source_dfs:
        source_dfs[alias] = pd.DataFrame(columns=df_ft.columns)

for alias in SOURCE_ORDER:
    print(f"{alias}: {len(source_dfs[alias])} rows")


In [ ]:
import torch
from transformers import pipeline, TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel


def build_source_splits(source_df: pd.DataFrame, seed: int = 42):
    data = source_df.copy()
    if len(data) < 10:
        raise ValueError(f"Dataset too small: {len(data)}")

    ds = Dataset.from_pandas(
        data[["question_fr", "answer", "prompt", "difficulty", "source_model_alias"]],
        preserve_index=False,
    )

    split1 = ds.train_test_split(test_size=0.2, seed=seed)
    split2 = split1["test"].train_test_split(test_size=0.5, seed=seed)
    return DatasetDict(
        train=split1["train"],
        validation=split2["train"],
        test=split2["test"],
    )


def clean_generated_sql(text: str) -> str:
    text = text or ""
    text = re.sub(r"```sql", "", text, flags=re.IGNORECASE)
    text = re.sub(r"```", "", text)
    text = text.split("<|im_end|>")[0]
    return text.strip()


def run_inference_on_split(model, tokenizer, eval_dataset, max_samples=80, batch_size=4):
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"

    pipe = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        device_map="auto",
    )

    n = min(len(eval_dataset), max_samples)
    examples = [eval_dataset[i] for i in range(n)]
    prompts = [ex["prompt"] for ex in examples]

    generated_sql = []
    for i in tqdm(range(0, len(prompts), batch_size), desc="Inference"):
        batch_prompts = prompts[i : i + batch_size]
        outputs = pipe(
            batch_prompts,
            max_new_tokens=256,
            do_sample=False,
            return_full_text=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        for out in outputs:
            txt = out[0]["generated_text"] if isinstance(out, list) else out["generated_text"]
            generated_sql.append(clean_generated_sql(txt))

    rows = []
    for i in range(n):
        rows.append(
            {
                "question_fr": examples[i]["question_fr"],
                "gold_sql": examples[i]["answer"],
                "generated_sql": generated_sql[i],
                "difficulty": examples[i].get("difficulty", "unknown"),
                "source_model_alias": examples[i].get("source_model_alias", "unknown"),
            }
        )
    return pd.DataFrame(rows)


def load_base_model_and_tokenizer(model_id: str):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_id,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=torch.bfloat16,
        load_in_4bit=LOAD_IN_4BIT,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return model, tokenizer


def train_ft_for_source(source_name: str, splits: DatasetDict):
    model, tokenizer = load_base_model_and_tokenizer(MODEL_ID)

    model = FastLanguageModel.get_peft_model(
        model,
        r=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_alpha=64,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
    )

    args = TrainingArguments(
        output_dir=str(FT_COMPARE_DIR / f"runs_{source_name}"),
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACC_STEPS,
        learning_rate=LEARNING_RATE,
        num_train_epochs=NUM_EPOCHS,
        logging_steps=20,
        save_strategy="no",
        bf16=True,
        fp16=False,
        optim="paged_adamw_8bit",
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        report_to="none",
    )

    sig = inspect.signature(SFTTrainer.__init__)
    trainer_kwargs = dict(
        model=model,
        args=args,
        train_dataset=splits["train"],
        eval_dataset=splits["validation"],
        formatting_func=format_prompt_train,
        packing=True,
    )
    if "tokenizer" in sig.parameters:
        trainer_kwargs["tokenizer"] = tokenizer
    elif "processing_class" in sig.parameters:
        trainer_kwargs["processing_class"] = tokenizer
    if "max_seq_length" in sig.parameters:
        trainer_kwargs["max_seq_length"] = MAX_SEQ_LENGTH

    trainer = SFTTrainer(**trainer_kwargs)
    trainer.train()

    adapter_dir = ADAPTERS_DIR / source_name
    trainer.save_model(str(adapter_dir))
    return model, tokenizer, adapter_dir


In [ ]:
def execute_sql_safely(conn, sql: str) -> str:
    try:
        stmt = (sql or "").strip().rstrip(";")
        if not stmt:
            return "Execution Error: Empty SQL"
        res = pd.read_sql_query(stmt, conn)
        if res.empty:
            return "Empty Result"
        return res.head(5).to_markdown(index=False)
    except Exception as e:
        return f"Execution Error: {str(e)}"


def judge_one_pair(question, gold_sql, gold_result, gen_sql, gen_result, schema_text, client, model_id):
    system_prompt = """
You are an expert SQL judge for SFIL municipal finance.
Return JSON only with:
{
  "code_quality_score": number,
  "relevance_gen": 0 or 1,
  "relevance_gold": 0 or 1,
  "logic_gen": 0 or 1,
  "logic_gold": 0 or 1,
  "fit_with_gold": 0 or 1,
  "reason": "short"
}
Rules:
- code_quality_score is from 0 to 3.
- If generated SQL has execution error, set relevance_gen=0, logic_gen=0, fit_with_gold=0.
- fit_with_gold compares values, not column names.
""".strip()

    user_prompt = f"""
QUESTION:
{question}

SCHEMA:
{schema_text[:8000]}

[GOLD SQL]
{gold_sql}

[GOLD RESULT]
{gold_result}

[GEN SQL]
{gen_sql}

[GEN RESULT]
{gen_result}
""".strip()

    try:
        resp = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.0,
            response_format={"type": "json_object"},
        )
        payload = json.loads(resp.choices[0].message.content)
        return {
            "code_quality_score": float(payload.get("code_quality_score", 0)),
            "relevance_gen": int(payload.get("relevance_gen", 0)),
            "relevance_gold": int(payload.get("relevance_gold", 0)),
            "logic_gen": int(payload.get("logic_gen", 0)),
            "logic_gold": int(payload.get("logic_gold", 0)),
            "fit_with_gold": int(payload.get("fit_with_gold", 0)),
            "judge_reason": str(payload.get("reason", "")).strip(),
            "judge_error": "",
        }
    except Exception as e:
        return {
            "code_quality_score": 0.0,
            "relevance_gen": 0,
            "relevance_gold": 0,
            "logic_gen": 0,
            "logic_gold": 0,
            "fit_with_gold": 0,
            "judge_reason": "",
            "judge_error": str(e),
        }


def evaluate_sql_df(pred_df, db_path, schema_text, client, model_id):
    conn = sqlite3.connect(db_path)
    out = pred_df.copy()

    out["gold_result_str"] = out["gold_sql"].apply(lambda x: execute_sql_safely(conn, x))
    out["gen_result_str"] = out["generated_sql"].apply(lambda x: execute_sql_safely(conn, x))

    judged = []
    for _, row in tqdm(out.iterrows(), total=len(out), desc="Judge"):
        judged.append(
            judge_one_pair(
                question=row["question_fr"],
                gold_sql=row["gold_sql"],
                gold_result=row["gold_result_str"],
                gen_sql=row["generated_sql"],
                gen_result=row["gen_result_str"],
                schema_text=schema_text,
                client=client,
                model_id=model_id,
            )
        )

    conn.close()
    judged_df = pd.DataFrame(judged)
    return pd.concat([out.reset_index(drop=True), judged_df.reset_index(drop=True)], axis=1)


In [ ]:
def plot_score_distributions(list_results_df, model_names, title_suffix=""):
    metrics = [
        "code_quality_score",
        "relevance_gen",
        "relevance_gold",
        "logic_gen",
        "logic_gold",
        "fit_with_gold",
    ]

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    colors = ["#4e79a7", "#f28e2b", "#e15759", "#76b7b2"]
    bar_width = 0.4

    for ax_idx, metric in enumerate(metrics):
        ax = axes[ax_idx]
        pretty_metric = metric.replace("_", " ").title()

        for i, dfm in enumerate(list_results_df):
            if metric not in dfm.columns:
                continue
            series = dfm[metric].dropna()
            if len(series) == 0:
                continue

            mean_val = series.mean()
            std_val = series.std() if len(series) > 1 else 0
            min_val = series.min()
            max_val = series.max()

            x = i
            color = colors[i % len(colors)]
            ax.plot([x, x], [min_val, max_val], color="black", linewidth=1.5, alpha=0.4, zorder=1)

            rect_bottom = mean_val - std_val
            rect_height = 2 * std_val
            ax.bar(
                x,
                rect_height,
                bottom=rect_bottom,
                width=bar_width,
                color=color,
                alpha=0.6,
                zorder=2,
            )

            ax.scatter(x, mean_val, color="white", edgecolor="black", s=80, linewidth=1.5, zorder=3)
            ax.text(x + 0.1, mean_val, f"{mean_val:.2f}", fontsize=9, va="center", fontweight="bold")

        ax.set_title(pretty_metric, fontsize=13, fontweight="bold", pad=10)
        ax.set_xticks(range(len(model_names)))
        ax.set_xticklabels(model_names, fontsize=10)
        if metric == "code_quality_score":
            ax.set_ylim(-1, 3.2)
        else:
            ax.set_ylim(-0.2, 1.2)

        ax.grid(axis="y", linestyle="--", alpha=0.3)
        ax.axhline(1.0, color="green", linestyle=":", alpha=0.3)

    plt.suptitle(f"Model Performance Comparison (Execution & Logic) {title_suffix}", fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()


In [ ]:
summary_rows = []

for source_name in SOURCE_ORDER:
    source_df = source_dfs[source_name].copy()

    if len(source_df) < 20:
        print(f"[SKIP] {source_name}: not enough rows ({len(source_df)})")
        continue

    print(f"
========== SOURCE: {source_name} | rows={len(source_df)} ==========")
    splits = build_source_splits(source_df, seed=SEED)

    # Base inference
    base_model, base_tokenizer = load_base_model_and_tokenizer(MODEL_ID)
    base_pred_df = run_inference_on_split(
        base_model,
        base_tokenizer,
        splits["test"],
        max_samples=MAX_EVAL_SAMPLES,
        batch_size=INFER_BATCH_SIZE,
    )

    eval_base_df = evaluate_sql_df(
        base_pred_df,
        DB_SQLITE_PATH,
        SCHEMA_SQL_TEXT,
        judge_client,
        JUDGE_MODEL_ID,
    )

    # FT inference
    eval_ft_df = None
    if ENABLE_TRAINING:
        ft_model, ft_tokenizer, adapter_dir = train_ft_for_source(source_name, splits)
        print(f"[FT] adapter saved -> {adapter_dir}")

        ft_pred_df = run_inference_on_split(
            ft_model,
            ft_tokenizer,
            splits["test"],
            max_samples=MAX_EVAL_SAMPLES,
            batch_size=INFER_BATCH_SIZE,
        )

        eval_ft_df = evaluate_sql_df(
            ft_pred_df,
            DB_SQLITE_PATH,
            SCHEMA_SQL_TEXT,
            judge_client,
            JUDGE_MODEL_ID,
        )

    out_dir = FT_COMPARE_DIR / source_name
    out_dir.mkdir(parents=True, exist_ok=True)

    eval_base_df.to_csv(out_dir / "evaluation_base.csv", index=False)
    eval_base_df.to_json(out_dir / "evaluation_base.jsonl", orient="records", lines=True, force_ascii=False)

    if eval_ft_df is not None:
        eval_ft_df.to_csv(out_dir / "evaluation_ft.csv", index=False)
        eval_ft_df.to_json(out_dir / "evaluation_ft.jsonl", orient="records", lines=True, force_ascii=False)
        plot_score_distributions([eval_ft_df, eval_base_df], ["ft_model", "base_model"], title_suffix=f"- {source_name}")

    row = {
        "source_name": source_name,
        "n_test": len(eval_base_df),
        "base_code_quality_score_mean": float(eval_base_df["code_quality_score"].mean()),
        "base_relevance_gen_mean": float(eval_base_df["relevance_gen"].mean()),
        "base_relevance_gold_mean": float(eval_base_df["relevance_gold"].mean()),
        "base_logic_gen_mean": float(eval_base_df["logic_gen"].mean()),
        "base_logic_gold_mean": float(eval_base_df["logic_gold"].mean()),
        "base_fit_with_gold_mean": float(eval_base_df["fit_with_gold"].mean()),
    }

    if eval_ft_df is not None:
        row.update(
            {
                "ft_code_quality_score_mean": float(eval_ft_df["code_quality_score"].mean()),
                "ft_relevance_gen_mean": float(eval_ft_df["relevance_gen"].mean()),
                "ft_relevance_gold_mean": float(eval_ft_df["relevance_gold"].mean()),
                "ft_logic_gen_mean": float(eval_ft_df["logic_gen"].mean()),
                "ft_logic_gold_mean": float(eval_ft_df["logic_gold"].mean()),
                "ft_fit_with_gold_mean": float(eval_ft_df["fit_with_gold"].mean()),
            }
        )

    summary_rows.append(row)

    # cleanup
    for obj_name in ["base_model", "base_tokenizer", "ft_model", "ft_tokenizer"]:
        if obj_name in locals():
            del locals()[obj_name]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(FT_COMPARE_DIR / "summary_all_sources.csv", index=False)
summary_df
